# RAG Aquila — Lancement sur Google Colab

Lance les cellules dans l'ordre. Active le GPU : `Runtime > Change runtime type > T4 GPU`

## Étape 1 — Clone du repo GitHub

In [ ]:
from google.colab import userdata
token = userdata.get('AH')

%cd /content
!rm -rf /content/RAG_Aquila
!git clone https://{token}@github.com/paulhenriduranton-collab/RAG_Aquila.git /content/RAG_Aquila
%cd /content/RAG_Aquila

## Étape 2 — Installation des dépendances

In [ ]:
!pip install -q langchain-chroma langchain-ollama langchain-core langgraph rank-bm25 sentence-transformers langchain-huggingface
print('Installation terminée.')

## Étape 3 — Remplacement d'Ollama par HuggingFace

Ollama tourne en local, pas dans Colab. On le remplace par des modèles HuggingFace équivalents.

In [ ]:
import sys
from types import ModuleType
import torch
from transformers import pipeline as hf_pipeline
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline

device = 0 if torch.cuda.is_available() else -1
device_str = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device_str}')

print('Chargement embeddings bge-m3 (~570 Mo)...')
_embeddings = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': device_str},
)

print('Chargement LLM Qwen2.5-1.5B (~3 Go)...')
_pipe = hf_pipeline(
    'text-generation',
    model='Qwen/Qwen2.5-1.5B-Instruct',
    max_new_tokens=512,
    do_sample=False,
    return_full_text=False,
    device=device,
)
_llm = HuggingFacePipeline(pipeline=_pipe)

class _FakeOllamaEmbeddings:
    def __new__(cls, model, **kwargs): return _embeddings
class _FakeOllamaLLM:
    def __new__(cls, **kwargs): return _llm

fake_ollama = ModuleType('langchain_ollama')
fake_ollama.OllamaEmbeddings = _FakeOllamaEmbeddings
fake_ollama.OllamaLLM = _FakeOllamaLLM
sys.modules['langchain_ollama'] = fake_ollama
print('Patch appliqué.')

## Étape 4 — Lancement

In [ ]:
import sys, os
from pathlib import Path

sys.path.insert(0, '/content/RAG_Aquila/src')
import ask
ask.VECTOR_DB_DIR = Path('/content/RAG_Aquila/vector_db')

import run_agentic_all
run_agentic_all.main()